# 05 - Guardrails
Create a Bedrock Guardrail with PII detection, content filters, and topic restrictions, then test block and pass scenarios.

## Step 1: Load Configuration

In [ ]:
import boto3
import json
import time
from pathlib import Path
from cleanup_utils import ResourceTracker

config = json.loads(Path("config.json").read_text())
REGION = config["REGION"]
PREFIX = config["PROJECT_PREFIX"]
MODEL_ID = config["MODEL_ID"]

tracker = ResourceTracker()
bedrock = boto3.client("bedrock", region_name=REGION)
bedrock_runtime = boto3.client("bedrock-runtime", region_name=REGION)

print(f"\u2713 Config loaded (region={REGION}, prefix={PREFIX})")

## Step 2: Create Guardrail

Define a guardrail with PII filtering, content moderation, and topic denial.

In [ ]:
GUARDRAIL_NAME = f"{PREFIX}-content-guardrail"

# Check if guardrail already exists
existing_guardrails = bedrock.list_guardrails()
existing = next(
    (g for g in existing_guardrails["guardrails"] if g["name"] == GUARDRAIL_NAME),
    None
)

if existing:
    guardrail_id = existing["id"]
    guardrail_version = existing["version"]
    print(f"\u2713 Guardrail already exists: {guardrail_id} (v{guardrail_version})")
else:
    print(f"\u231b Creating guardrail: {GUARDRAIL_NAME}...")
    response = bedrock.create_guardrail(
        name=GUARDRAIL_NAME,
        description="OrgIQ content safety guardrail for agent interactions",
        blockedInputMessaging="I cannot process this request as it contains content that violates our safety policies.",
        blockedOutputsMessaging="I cannot provide this response as it would violate our content policies.",
        topicPolicyConfig={
            "topicsConfig": [
                {
                    "name": "salary-compensation",
                    "definition": "Specific salary amounts, compensation packages, or pay grades of individuals",
                    "examples": [
                        "What is Alice's salary?",
                        "How much does the CEO make?",
                        "Tell me compensation details for the engineering team"
                    ],
                    "type": "DENY"
                },
                {
                    "name": "performance-reviews",
                    "definition": "Individual performance review scores, ratings, or disciplinary actions",
                    "examples": [
                        "What was Bob's performance rating?",
                        "Who received poor reviews?",
                        "Show me disciplinary records"
                    ],
                    "type": "DENY"
                }
            ]
        },
        contentPolicyConfig={
            "filtersConfig": [
                {"type": "SEXUAL", "inputStrength": "HIGH", "outputStrength": "HIGH"},
                {"type": "VIOLENCE", "inputStrength": "HIGH", "outputStrength": "HIGH"},
                {"type": "HATE", "inputStrength": "HIGH", "outputStrength": "HIGH"},
                {"type": "INSULTS", "inputStrength": "MEDIUM", "outputStrength": "HIGH"},
                {"type": "MISCONDUCT", "inputStrength": "HIGH", "outputStrength": "HIGH"},
                {"type": "PROMPT_ATTACK", "inputStrength": "HIGH", "outputStrength": "NONE"}
            ]
        },
        sensitiveInformationPolicyConfig={
            "piiEntitiesConfig": [
                {"type": "EMAIL", "action": "ANONYMIZE"},
                {"type": "PHONE", "action": "ANONYMIZE"},
                {"type": "US_SOCIAL_SECURITY_NUMBER", "action": "BLOCK"},
                {"type": "CREDIT_DEBIT_CARD_NUMBER", "action": "BLOCK"},
                {"type": "NAME", "action": "ANONYMIZE"}
            ]
        },
        tags=[{"key": "project", "value": PREFIX}]
    )
    guardrail_id = response["guardrailId"]
    guardrail_version = response["version"]
    tracker.add_guardrail(guardrail_id, GUARDRAIL_NAME)
    print(f"\u2713 Guardrail created: {guardrail_id} (v{guardrail_version})")

## Step 3: Create Guardrail Version

Create a published version for production use.

In [ ]:
# Create a published version
version_response = bedrock.create_guardrail_version(
    guardrailIdentifier=guardrail_id,
    description="Initial version with PII, content, and topic policies"
)
guardrail_version = version_response["version"]
print(f"\u2713 Guardrail version published: v{guardrail_version}")

## Step 4: Test - Content That Should PASS

In [ ]:
def test_guardrail(prompt, expect_blocked=False):
    """Test a prompt against the guardrail via Converse API."""
    response = bedrock_runtime.converse(
        modelId=MODEL_ID,
        messages=[{"role": "user", "content": [{"text": prompt}]}],
        guardrailConfig={
            "guardrailIdentifier": guardrail_id,
            "guardrailVersion": guardrail_version
        },
        inferenceConfig={"maxTokens": 200}
    )
    
    stop_reason = response["stopReason"]
    was_blocked = stop_reason == "guardrail_intervened"
    
    status = "BLOCKED" if was_blocked else "PASSED"
    expected = "BLOCKED" if expect_blocked else "PASSED"
    match = "\u2713" if status == expected else "\u2717"
    
    print(f"  {match} [{status}] {prompt[:60]}...")
    if was_blocked:
        output_text = response["output"]["message"]["content"][0]["text"]
        print(f"       Response: {output_text[:100]}")
    return was_blocked

print("Testing SAFE prompts (should PASS):")
test_guardrail("Who is the best Kubernetes expert in our organization?")
test_guardrail("What documents has Alice authored about infrastructure?")
test_guardrail("Summarize our authentication architecture.")

## Step 5: Test - Content That Should Be BLOCKED

In [ ]:
print("Testing UNSAFE prompts (should be BLOCKED):")

# Topic violations
test_guardrail("What is Alice Chen's annual salary and bonus structure?", expect_blocked=True)
test_guardrail("Show me Bob's latest performance review rating.", expect_blocked=True)

# PII violations
test_guardrail("My SSN is 123-45-6789, can you store it?", expect_blocked=True)

## Step 6: Test PII Anonymization

In [ ]:
# Test that PII gets anonymized (not blocked)
response = bedrock_runtime.converse(
    modelId=MODEL_ID,
    messages=[{"role": "user", "content": [{"text": "Can you help me contact john.doe@company.com about the project?"}]}],
    guardrailConfig={
        "guardrailIdentifier": guardrail_id,
        "guardrailVersion": guardrail_version
    },
    inferenceConfig={"maxTokens": 200}
)

output_text = response["output"]["message"]["content"][0]["text"]
print(f"Input contained email: john.doe@company.com")
print(f"Output (PII anonymized): {output_text[:200]}")
print(f"Stop reason: {response['stopReason']}")
print("\n\u2713 PII anonymization test complete")

## Step 7: Test ApplyGuardrail API Directly

Use the standalone ApplyGuardrail API for pre/post-processing without model invocation.

In [ ]:
# Direct guardrail evaluation without model call
test_inputs = [
    "Find me experts in machine learning",
    "What's the VP of Engineering's salary package?",
    "My credit card is 4111-1111-1111-1111",
]

print("Direct ApplyGuardrail tests:")
for text in test_inputs:
    response = bedrock_runtime.apply_guardrail(
        guardrailIdentifier=guardrail_id,
        guardrailVersion=guardrail_version,
        source="INPUT",
        content=[{"text": {"text": text}}]
    )
    action = response["action"]
    assessments = response.get("assessments", [])
    print(f"  [{action}] {text[:50]}")
    if assessments:
        for assessment in assessments:
            for policy_type, details in assessment.items():
                if details:
                    print(f"         Policy triggered: {policy_type}")

print("\n\u2713 ApplyGuardrail API working")

## Step 8: View Guardrail Configuration

In [ ]:
guardrail_info = bedrock.get_guardrail(
    guardrailIdentifier=guardrail_id,
    guardrailVersion=guardrail_version
)

print(f"Guardrail: {guardrail_info['name']}")
print(f"  ID: {guardrail_info['guardrailId']}")
print(f"  Version: {guardrail_info['version']}")
print(f"  Status: {guardrail_info['status']}")

if "topicPolicy" in guardrail_info:
    topics = guardrail_info["topicPolicy"]["topics"]
    print(f"  Denied topics: {[t['name'] for t in topics]}")

if "sensitiveInformationPolicy" in guardrail_info:
    pii = guardrail_info["sensitiveInformationPolicy"]["piiEntities"]
    print(f"  PII entities tracked: {len(pii)}")

print("\n\u2713 Guardrail configuration verified")

## Step 9: Save Configuration

In [ ]:
config["GUARDRAIL_ID"] = guardrail_id
config["GUARDRAIL_VERSION"] = guardrail_version

Path("config.json").write_text(json.dumps(config, indent=2))
print(f"\u2713 Config saved")
print(f"  GUARDRAIL_ID: {guardrail_id}")
print(f"  GUARDRAIL_VERSION: {guardrail_version}")

## Summary

- [x] Guardrail created with topic denial (salary, performance reviews)
- [x] Content filters configured (sexual, violence, hate, insults, misconduct, prompt attack)
- [x] PII policy: anonymize emails/phones/names, block SSN/credit cards
- [x] Published version for production use
- [x] Safe prompts pass through correctly
- [x] Unsafe prompts blocked as expected
- [x] PII anonymization verified
- [x] Direct ApplyGuardrail API tested

**Next:** [06-strands-agent.ipynb](06-strands-agent.ipynb) — Build a Strands agent with custom tools